# Aviation Accidents Analysis

You are part of a consulting firm tasked with analysing commercial and passenger jet airline safety.
The client (an airline/airplane insurer) wants to know which aircraft **makes/models** exhibit:

- **Low rates of total destruction** in the event of an accident  
- **Low likelihood of fatal or serious passenger injuries**

They also want general insight into conditions that affect safety outcomes.

**Scope:**
- Professional builds only (no amateur-built aircraft)  
- Data from 1983 onwards (40-year max lifetime filter)  
- Separate recommendations for small (<= 20 total passengers) and large (> 20) aircraft  
- Statistically robust comparisons (sufficient sample sizes enforced)

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 60)

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df_raw = pd.read_csv('AviationData.csv', encoding='latin-1', low_memory=False)
print("Shape:", df_raw.shape)
df_raw.head(3)

Shape: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007


In [3]:
# Data types
df_raw.dtypes

Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                      str
Longitude                     str
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Broad.phase.of.flight         str
Report.Status 

In [4]:
# NaN counts per column
nan_counts = df_raw.isnull().sum().sort_values(ascending=False)
nan_pct = (nan_counts / len(df_raw) * 100).round(1)
nan_report = pd.DataFrame({'NaN_count': nan_counts, 'NaN_%': nan_pct})
print(nan_report)

                        NaN_count  NaN_%
Schedule                    76307   85.8
Air.carrier                 72241   81.3
FAR.Description             56866   64.0
Aircraft.Category           56602   63.7
Longitude                   54516   61.3
Latitude                    54507   61.3
Airport.Code                38757   43.6
Airport.Name                36185   40.7
Broad.phase.of.flight       27165   30.6
Publication.Date            13771   15.5
Total.Serious.Injuries      12510   14.1
Total.Minor.Injuries        11933   13.4
Total.Fatal.Injuries        11401   12.8
Engine.Type                  7096    8.0
Report.Status                6384    7.2
Purpose.of.flight            6192    7.0
Number.of.Engines            6084    6.8
Total.Uninjured              5912    6.7
Weather.Condition            4492    5.1
Aircraft.damage              3194    3.6
Registration.Number          1382    1.6
Injury.Severity              1000    1.1
Country                       226    0.3
Amateur.Built   

In [5]:
# Numeric summary statistics
df_raw.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We filter to the scope the client is interested in:

1. **Investigation type = Accident** – we exclude mere incidents (near-misses, etc.)  
2. **Amateur.Built = No** – only professional builds  
3. **Event.Date >= 1983** – 40-year maximum lifetime filter  
4. **Aircraft.Category = Airplane (or NaN)** – Aircraft.Category has >63 % missing values;
   the known-category records are predominantly Airplanes.  
   We keep rows where the category is `Airplane` or is missing (helicopter-specific makes such as
   Robinson/Sikorsky will be removed during Make cleaning anyway).


In [6]:
# Parse date
df_raw['Event.Date'] = pd.to_datetime(df_raw['Event.Date'], errors='coerce')

# --- Filter 1: Accidents only ---
df = df_raw[df_raw['Investigation.Type'] == 'Accident'].copy()
print("After accident filter:", df.shape)

# --- Filter 2: Professional builds ---
df = df[df['Amateur.Built'] == 'No'].copy()
print("After amateur-built filter:", df.shape)

# --- Filter 3: 1983 onwards ---
df = df[df['Event.Date'].dt.year >= 1983].copy()
print("After 1983 filter:", df.shape)

# --- Filter 4: Airplanes only (Airplane or NaN category) ---
df = df[(df['Aircraft.Category'] == 'Airplane') | (df['Aircraft.Category'].isna())].copy()
print("After airplane category filter:", df.shape)

After accident filter: (85015, 31)
After amateur-built filter:

 (76520, 31)
After 1983 filter: (73286, 31)
After airplane category filter: (69422, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are the client's primary interest.  
We clean the injury columns and then build two derived metrics:

1. **`injury_fraction`** – fraction of passengers fatally or seriously injured  
2. **`is_destroyed`** – binary flag indicating the aircraft was destroyed

**Construct metric for fatal/serious injuries**

*Strategy:* Fill all injury count NaNs with 0 — if no injuries were recorded it is reasonable to
assume zero (incidents with large numbers of casualties are well-documented).  
Total on-board passengers = fatal + serious + minor + uninjured.  
`injury_fraction` = (fatal + serious) / total_passengers.  
Where total_passengers = 0 (extremely rare), we set `injury_fraction` = 0.


In [7]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
               'Total.Minor.Injuries', 'Total.Uninjured']

# Fill NaN with 0 before arithmetic
df[injury_cols] = df[injury_cols].fillna(0)

# Total estimated passengers on board
df['total_passengers'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] +
                           df['Total.Minor.Injuries'] + df['Total.Uninjured'])

# Fraction fatally or seriously injured (0 when nobody on board recorded)
df['injury_fraction'] = np.where(
    df['total_passengers'] > 0,
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['total_passengers'],
    0.0
)

print("injury_fraction statistics:")
print(df['injury_fraction'].describe())
print()
print("Rows with total_passengers == 0:", (df['total_passengers'] == 0).sum())

injury_fraction statistics:
count    69422.000000
mean         0.280281
std          0.432082
min          0.000000
25%          0.000000
50%          0.000000
75%          0.833333
max          1.000000
Name: injury_fraction, dtype: float64

Rows with total_passengers == 0: 371


**Aircraft.Damage**
- `Unknown` values are treated as NaN and will not contribute to the destroyed-rate calculations  
- A new binary column `is_destroyed` is created (1 = Destroyed, 0 = anything else with a known value)


In [8]:
print("Aircraft.damage value counts (raw):")
print(df['Aircraft.damage'].value_counts(dropna=False))

Aircraft.damage value counts (raw):
Aircraft.damage
Substantial    52511
Destroyed      14952
NaN             1285
Minor            599
Unknown           75
Name: count, dtype: int64


In [9]:
# Standardise capitalisation
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()

# Replace 'Unknown' with NaN — we cannot determine outcome
df.loc[df['Aircraft.damage'] == 'Unknown', 'Aircraft.damage'] = np.nan

# Binary destroyed flag
df['is_destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(float)
# Keep NaN for missing damage info
df.loc[df['Aircraft.damage'].isna(), 'is_destroyed'] = np.nan

print("Cleaned Aircraft.damage counts:")
print(df['Aircraft.damage'].value_counts(dropna=False))
print()
print("is_destroyed distribution:")
print(df['is_destroyed'].value_counts(dropna=False))

Cleaned Aircraft.damage counts:
Aircraft.damage
Substantial    52511
Destroyed      14952
NaN             1360
Minor            599
Name: count, dtype: int64

is_destroyed distribution:
is_destroyed
0.0    53110
1.0    14952
NaN     1360
Name: count, dtype: int64


### Investigate the *Make* column

**Cleaning tasks identified:**
1. Strip whitespace and standardise to Title Case  
2. Consolidate known duplicates (e.g., 'Cessna', 'Cessna Aircraft', 'Cessna Aircraft Co')  
3. Keep only makes with **≥ 50** accident records (sufficient statistical power)


In [10]:
# Before cleaning
print("Unique Makes (raw):", df['Make'].nunique())
print(df['Make'].value_counts().head(40))

Unique Makes (raw): 1829
Make
Cessna                            20590
Piper                             11138
CESSNA                             4820
Beech                              3910
PIPER                              2797
Bell                               1743
Mooney                             1012
BEECH                              1007
Grumman                             974
Boeing                              862
Bellanca                            816
Hughes                              680
Robinson                            656
Air Tractor                         575
Schweizer                           542
Aeronca                             457
BOEING                              444
Maule                               425
Champion                            412
Aero Commander                      335
De Havilland                        329
Stinson                             327
Rockwell                            311
Taylorcraft                         304
Luscombe  

In [11]:
# Step 1: strip & title-case
df['Make'] = df['Make'].str.strip().str.title()

# Step 2: consolidate known variant spellings
make_map = {
    # Cessna variants
    'Cessna Aircraft': 'Cessna',
    'Cessna Aircraft Co': 'Cessna',
    'Cessna Aircraft Company': 'Cessna',
    # Piper variants
    'Piper Aircraft': 'Piper',
    'Piper Aircraft Corp': 'Piper',
    'Piper Aircraft Inc': 'Piper',
    # Beech variants
    'Beechcraft': 'Beech',
    'Beech Aircraft': 'Beech',
    'Beech Aircraft Corp': 'Beech',
    'Beech Aircraft Corporation': 'Beech',
    # Boeing variants
    'Boeing Aircraft': 'Boeing',
    'Boeing Company': 'Boeing',
    # McDonnell Douglas variants
    'Mcdonnell Douglas': 'McDonnell Douglas',
    'Mc Donnell Douglas': 'McDonnell Douglas',
    # Airbus variants
    'Airbus Industrie': 'Airbus',
    'Airbus S.A.S.': 'Airbus',
    # Air Tractor variants
    'Air Tractor Inc': 'Air Tractor',
    'Air Tractor, Inc': 'Air Tractor',
    # Cirrus variants
    'Cirrus Design': 'Cirrus',
    'Cirrus Design Corp': 'Cirrus',
    'Cirrus Design Corporation': 'Cirrus',
    # Embraer variants
    'Embraer S.A.': 'Embraer',
    'Embraer - Empresa Brasileira De Aeronautica S.A.': 'Embraer',
    # Grumman variants
    'Grumman American': 'Grumman',
    'Grumman Corp': 'Grumman',
    # Bell variants
    'Bell Helicopter': 'Bell',
    'Bell Helicopter Textron': 'Bell',
    # Rockwell variants
    'Rockwell International': 'Rockwell',
    'Rockwell International Corp': 'Rockwell',
    # Socata variants
    'Socata - Groupe Aerospatiale': 'Socata',
    # Bombardier
    'Bombardier Inc': 'Bombardier',
    # De Havilland
    'De Havilland Canada': 'De Havilland',
    'De Havilland Aircraft': 'De Havilland',
}

df['Make'] = df['Make'].replace(make_map)

print("Unique Makes after consolidation:", df['Make'].nunique())
print()
print("Top 30 makes post-cleaning:")
print(df['Make'].value_counts().head(30))

Unique Makes after consolidation: 1491

Top 30 makes post-cleaning:
Make
Cessna               25453
Piper                13969
Beech                 4949
Bell                  1757
Boeing                1307
Grumman               1261
Mooney                1249
Bellanca               973
Air Tractor            888
Hughes                 681
Robinson               674
Aeronca                606
Maule                  569
Schweizer              547
Champion               502
Stinson                418
Aero Commander         404
Rockwell               404
Luscombe               390
Cirrus                 371
Taylorcraft            366
North American         364
De Havilland           362
McDonnell Douglas      336
Hiller                 279
Aerospatiale           264
Ayres                  230
Enstrom                207
Douglas                183
Airbus                 169
Name: count, dtype: int64


In [12]:
# Step 3: Keep makes with >= 50 records
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
print(f"Makes with >= 50 records: {len(valid_makes)}")

df = df[df['Make'].isin(valid_makes)].copy()
print("Rows retained:", df.shape[0])

Makes with >= 50 records: 75
Rows retained: 64226


### Inspect Model column
- Drop rows with NaN Model  
- Model labels may not be unique across makes (e.g., '172' exists for both Cessna and possibly others)  
- Create `Make_Model` as a unique identifier


In [13]:
print("Model NaN count:", df['Model'].isna().sum())
df = df.dropna(subset=['Model']).copy()
print("Rows after dropping NaN models:", df.shape[0])

Model NaN count: 15
Rows after dropping NaN models: 64211


In [14]:
# Check whether model labels are unique to a make
model_make_counts = df.groupby('Model')['Make'].nunique()
shared_models = model_make_counts[model_make_counts > 1]
print(f"Model labels shared across multiple makes: {len(shared_models)}")
print(shared_models.sort_values(ascending=False).head(10))

Model labels shared across multiple makes: 331
Model
500       6
G-164A    6
G-164C    5
G-164B    5
AA-5      4
8KCAB     4
G-164     4
7ECA      4
7EC       4
7AC       4
Name: Make, dtype: int64


In [15]:
# Create unique identifier: Make_Model
df['Make_Model'] = df['Make'].str.strip() + ' ' + df['Model'].str.strip()
print("Sample Make_Model values:")
print(df['Make_Model'].value_counts().head(20))

Sample Make_Model values:
Make_Model
Cessna 152         2216
Cessna 172         1641
Cessna 172N        1091
Piper PA-28-140     862
Cessna 172M         754
Cessna 150          745
Cessna 172P         661
Cessna 182          611
Cessna 180          594
Piper PA-18         550
Piper PA-18-150     549
Cessna 150M         549
Piper PA-28-180     546
Piper PA-28-161     522
Piper PA-28-181     508
Cessna 150L         433
Piper PA-38-112     422
Beech A36           406
Cessna 140          384
Cessna 170B         374
Name: count, dtype: int64


### Cleaning other columns

We inspect and clean the following contextual columns:
- `Engine.Type`
- `Weather.Condition`
- `Number.of.Engines`
- `Purpose.of.flight`
- `Broad.phase.of.flight`

NaN values are **not imputed** — they will be excluded from group-level analyses.


In [16]:
# ── Engine.Type ──────────────────────────────────────────────
print("Engine.Type (raw):")
print(df['Engine.Type'].value_counts(dropna=False))

# Normalise capitalisation, collapse low-frequency / ambiguous labels
df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()
df['Engine.Type'] = df['Engine.Type'].replace({
    'Unk': np.nan,
    'Unknown': np.nan,
    'None': np.nan,
    'Lr': np.nan,
})
print()
print("Engine.Type (cleaned):")
print(df['Engine.Type'].value_counts(dropna=False))

Engine.Type (raw):
Engine.Type
Reciprocating    53409
NaN               3515
Turbo Prop        2466
Turbo Shaft       2021
Unknown           1345
Turbo Fan         1146
Turbo Jet          308
UNK                  1
Name: count, dtype: int64



Engine.Type (cleaned):
Engine.Type
Reciprocating    53409
NaN               4861
Turbo Prop        2466
Turbo Shaft       2021
Turbo Fan         1146
Turbo Jet          308
Name: count, dtype: int64


In [17]:
# ── Weather.Condition ────────────────────────────────────────
print("Weather.Condition (raw):")
print(df['Weather.Condition'].value_counts(dropna=False))

df['Weather.Condition'] = df['Weather.Condition'].str.strip().str.upper()
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': np.nan})
print()
print("Weather.Condition (cleaned):")
print(df['Weather.Condition'].value_counts(dropna=False))

Weather.Condition (raw):
Weather.Condition
VMC    56429
IMC     4956
NaN     2064
UNK      607
Unk      155
Name: count, dtype: int64

Weather.Condition (cleaned):
Weather.Condition
VMC    56429
IMC     4956
NaN     2826
Name: count, dtype: int64


In [18]:
# ── Number.of.Engines ────────────────────────────────────────
print("Number.of.Engines distribution:")
print(df['Number.of.Engines'].value_counts(dropna=False).head(10))

# 0-engine records are almost certainly data errors; replace with NaN
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)

# Cap at 4 — record counts above 4 are negligible noise
df.loc[df['Number.of.Engines'] > 4, 'Number.of.Engines'] = np.nan
print()
print("Number.of.Engines (cleaned):")
print(df['Number.of.Engines'].value_counts(dropna=False))

Number.of.Engines distribution:
Number.of.Engines
1.0    52138
2.0     7938
NaN     3260
0.0      501
4.0      199
3.0      175
Name: count, dtype: int64

Number.of.Engines (cleaned):
Number.of.Engines
1.0    52138
2.0     7938
NaN     3761
4.0      199
3.0      175
Name: count, dtype: int64


In [19]:
# ── Purpose.of.flight ────────────────────────────────────────
print("Purpose.of.flight (raw - top 15):")
print(df['Purpose.of.flight'].value_counts(dropna=False).head(15))

df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({'Unknown': np.nan})
print()
print("Purpose.of.flight (cleaned):")
print(df['Purpose.of.flight'].value_counts(dropna=False).head(12))

Purpose.of.flight (raw - top 15):
Purpose.of.flight
Personal               36046
Instructional           8780
Unknown                 4230
Aerial Application      3895
Business                3224
NaN                     3163
Positioning             1221
Other Work Use           885
Public Aircraft          608
Ferry                    607
Aerial Observation       578
Executive/corporate      374
Skydiving                174
Flight Test              125
Banner Tow                97
Name: count, dtype: int64

Purpose.of.flight (cleaned):
Purpose.of.flight
Personal               36046
Instructional           8780
NaN                     7393
Aerial Application      3895
Business                3224
Positioning             1221
Other Work Use           885
Public Aircraft          608
Ferry                    607
Aerial Observation       578
Executive/Corporate      374
Skydiving                174
Name: count, dtype: int64


In [20]:
# ── Broad.phase.of.flight ────────────────────────────────────
print("Broad.phase.of.flight (raw):")
print(df['Broad.phase.of.flight'].value_counts(dropna=False).head(15))

df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title()
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({'Unknown': np.nan})
print()
print("Broad.phase.of.flight (cleaned):")
print(df['Broad.phase.of.flight'].value_counts(dropna=False).head(12))

Broad.phase.of.flight (raw):
Broad.phase.of.flight
NaN            16406
Landing        12543
Takeoff         9410
Cruise          8014
Maneuvering     6095
Approach        4957
Taxi            1499
Climb           1493
Descent         1443
Go-around       1169
Standing         725
Unknown          384
Other             73
Name: count, dtype: int64

Broad.phase.of.flight (cleaned):
Broad.phase.of.flight
NaN            16790
Landing        12543
Takeoff         9410
Cruise          8014
Maneuvering     6095
Approach        4957
Taxi            1499
Climb           1493
Descent         1443
Go-Around       1169
Standing         725
Other             73
Name: count, dtype: int64


### Column Removal
- Inspect the dataframe and drop any columns that have too many NaNs

In [21]:
nan_pct_final = df.isnull().mean().sort_values(ascending=False)
print("NaN fraction per column:")
print((nan_pct_final * 100).round(1))

NaN fraction per column:
Schedule                  87.1
Air.carrier               84.5
FAR.Description           73.0
Aircraft.Category         72.9
Longitude                 64.7
Latitude                  64.7
Airport.Code              42.3
Airport.Name              39.6
Broad.phase.of.flight     26.1
Publication.Date          17.7
Purpose.of.flight         11.5
Engine.Type                7.6
Number.of.Engines          5.9
Report.Status              5.2
Weather.Condition          4.4
is_destroyed               1.8
Aircraft.damage            1.8
Registration.Number        1.3
Injury.Severity            0.4
Country                    0.3
Location                   0.1
Event.Id                   0.0
Investigation.Type         0.0
Accident.Number            0.0
Amateur.Built              0.0
Make                       0.0
Event.Date                 0.0
Model                      0.0
Total.Minor.Injuries       0.0
Total.Serious.Injuries     0.0
Total.Fatal.Injuries       0.0
Total.Uninjure

In [22]:
# Drop columns with > 70% NaN — too sparse to be useful
drop_cols = nan_pct_final[nan_pct_final > 0.70].index.tolist()
print("Columns to drop (>70% NaN):", drop_cols)
df = df.drop(columns=drop_cols)
print("DataFrame shape after column removal:", df.shape)

Columns to drop (>70% NaN): ['Schedule', 'Air.carrier', 'FAR.Description', 'Aircraft.Category']
DataFrame shape after column removal: (64211, 31)


### Save DataFrame to csv

In [23]:
df.to_csv('aviation_cleaned.csv', index=False)
print("Saved aviation_cleaned.csv")
print("Final shape:", df.shape)
df.head(3)

Saved aviation_cleaned.csv
Final shape: (64211, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,Purpose.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,total_passengers,injury_fraction,is_destroyed,Make_Model
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,NaN,NaN,ONP,NEWPORT MUNICIPAL,Non-Fatal,Substantial,N1296M,Cessna,182P,No,1.0,Reciprocating,Personal,0.0,0.0,1.0,3.0,VMC,Approach,Probable Cause,NaN,4.0,0.0,0.0,Cessna 182P
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,NaN,NaN,3YR,MUNICIPAL,Non-Fatal,Substantial,N2639C,Cessna,182RG,No,1.0,Reciprocating,Personal,0.0,0.0,0.0,2.0,VMC,Landing,Probable Cause,NaN,2.0,0.0,0.0,Cessna 182RG
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,NaN,NaN,78Y,RANKIN,Non-Fatal,Substantial,N58664,Cessna,182P,No,1.0,Reciprocating,Personal,0.0,0.0,0.0,1.0,VMC,Takeoff,Probable Cause,NaN,1.0,0.0,0.0,Cessna 182P
